# VLM-13b: Diagnostic - Évaluation Phase 1

**Objectif**: Évaluer le modèle Phase 1 (projecteur seulement) sur le test set pour déterminer si Phase 2 a causé une régression.

**Contexte Critique**:
- Phase 2 accuracy: **12.52%** (catastrophique)
- Baseline accuracy: **40%**
- Phase 2 prédit presque tout comme "Mosaic Disease"

**Question**: Phase 1 était-il meilleur?

**Décisions selon résultats**:
- Si Phase 1 > 40%: Utiliser Phase 1 comme modèle final, Phase 2 a causé régression
- Si Phase 1 entre 12-40%: Les deux phases ont des problèmes
- Si Phase 1 < 12%: Le problème a commencé avant Phase 2

---

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from torch.utils.data import DataLoader
import json
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import gc

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 1. Configuration

In [ ]:
# Paths
base_dir = Path('/home/hounfodjidagba/learning/cassava')

# Phase 1 checkpoint (projecteur alignment)
phase1_checkpoint_dir = base_dir / 'models' / 'checkpoints' / 'phase1_projector' / 'best'

# Test dataset
test_jsonl_path = base_dir / 'outputs' / 'vlm_dataset' / 'test.jsonl'

# Output directory
output_dir = base_dir / 'outputs' / 'phase1_evaluation'
output_dir.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Class names
class_names = [
    "Bacterial Blight (CBB)",
    "Brown Streak Disease (CBSD)",
    "Green Mottle (CGM)",
    "Healthy",
    "Mosaic Disease (CMD)"
]

print(f"Phase 1 checkpoint: {phase1_checkpoint_dir}")
print(f"Test dataset: {test_jsonl_path}")
print(f"Output directory: {output_dir}")

## 2. Load Test Dataset

Using the same dataset as Notebook 13 for fair comparison.

In [ ]:
class CassavaVLMDataset(torch.utils.data.Dataset):
    """Dataset for VLM evaluation."""
    
    def __init__(self, jsonl_path: Path, processor, base_dir: Path):
        self.processor = processor
        self.base_dir = base_dir
        
        # Load JSONL
        self.samples = []
        with open(jsonl_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
        
        print(f"Loaded {len(self.samples)} samples from {jsonl_path}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load image
        image_path = self.base_dir / sample['image']
        image = Image.open(image_path).convert('RGB')
        
        # Get conversations
        messages = sample['conversations']
        
        # Get label from assistant response
        label = sample['label']
        
        # Format as processor expects
        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Process inputs
        inputs = self.processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True
        )
        
        # Remove batch dimension
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['label'] = label
        
        return inputs

print("Dataset class defined")

## 3. Load Phase 1 Model

**Important**: Loading the Phase 1 checkpoint (projecteur alignment only).

In [ ]:
print("Loading Phase 1 model (projector alignment)...")
print(f"Checkpoint: {phase1_checkpoint_dir}")

# Load processor first
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Processor loaded")

# Load Phase 1 model
model_phase1 = Qwen2VLForConditionalGeneration.from_pretrained(
    phase1_checkpoint_dir,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model_phase1.eval()
print("Phase 1 model loaded successfully")

# Memory status
if torch.cuda.is_available():
    print(f"\nGPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 4. Create Test DataLoader

In [ ]:
# Create test dataset
test_dataset = CassavaVLMDataset(
    jsonl_path=test_jsonl_path,
    processor=processor,
    base_dir=base_dir
)

# Create DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"\nTest dataset size: {len(test_dataset)}")
print(f"Number of batches: {len(test_loader)}")
print(f"Batch size: 2")

## 5. Evaluation Function

Using the same parsing logic as Notebook 13.

In [ ]:
def parse_disease_from_response(response: str) -> int:
    """
    Extract disease class from generated text response.
    
    Returns:
        Class ID (0-4) or -1 if parsing failed
    """
    disease_map = {
        'bacterial blight': 0,
        'brown streak disease': 1,
        'brown streak': 1,  # Alias
        'green mottle': 2,
        'healthy': 3,
        'mosaic disease': 4,
        'mosaic': 4  # Alias
    }
    
    response_lower = response.lower().strip()
    
    # Try to find disease names (longest first to avoid partial matches)
    for disease, class_id in sorted(disease_map.items(), key=lambda x: -len(x[0])):
        if disease in response_lower:
            return class_id
    
    return -1  # Parsing failed


@torch.no_grad()
def evaluate_with_generation(model, test_loader, processor, device):
    """
    Evaluate model using text generation and parsing.
    
    Returns:
        all_preds: List of predicted class IDs
        all_labels: List of true class IDs
        all_responses: List of generated text responses
        parsing_errors: Number of responses that couldn't be parsed
    """
    model.eval()
    
    all_preds = []
    all_labels = []
    all_responses = []
    parsing_errors = 0
    
    progress_bar = tqdm(test_loader, desc="Evaluating Phase 1")
    
    for batch in progress_bar:
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device, dtype=torch.bfloat16)
        image_grid_thw = batch['image_grid_thw'].to(device)
        true_labels = batch['label']
        
        # Generate responses
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,
            max_new_tokens=100,
            do_sample=False  # Greedy decoding for consistency
        )
        
        # Decode responses
        responses = processor.batch_decode(outputs, skip_special_tokens=True)
        
        # Parse and collect results
        for response, true_label in zip(responses, true_labels):
            pred_class = parse_disease_from_response(response)
            
            if pred_class == -1:
                parsing_errors += 1
                # Default to most common class (Healthy = 3) for failed parses
                pred_class = 3
            
            all_preds.append(pred_class)
            all_labels.append(true_label.item())
            all_responses.append(response)
        
        # Update progress
        progress_bar.set_postfix({
            'samples': len(all_preds),
            'parse_errors': parsing_errors
        })
    
    return all_preds, all_labels, all_responses, parsing_errors

print("Evaluation functions defined")

## 6. Run Evaluation on Phase 1

In [ ]:
print("Starting Phase 1 evaluation...\n")

# Evaluate
all_preds, all_labels, all_responses, parsing_errors = evaluate_with_generation(
    model=model_phase1,
    test_loader=test_loader,
    processor=processor,
    device=device
)

print(f"\nEvaluation complete!")
print(f"Total samples: {len(all_preds)}")
print(f"Parsing errors: {parsing_errors} ({parsing_errors/len(all_preds)*100:.1f}%)")

## 7. Calculate Metrics

In [ ]:
# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')

print("="*70)
print("PHASE 1 EVALUATION RESULTS")
print("="*70)
print(f"\nAccuracy:  {accuracy*100:.2f}%")
print(f"F1-Score:  {f1:.3f}")
print(f"\nParsing errors: {parsing_errors}/{len(all_preds)} ({parsing_errors/len(all_preds)*100:.1f}%)")

# Comparison with Phase 2
phase2_accuracy = 0.1252  # From Notebook 13
baseline_accuracy = 0.40  # Zero-shot baseline

print("\n" + "="*70)
print("COMPARISON")
print("="*70)
print(f"Baseline (zero-shot):  {baseline_accuracy*100:.2f}%")
print(f"Phase 1 (projector):   {accuracy*100:.2f}%")
print(f"Phase 2 (LoRA):        {phase2_accuracy*100:.2f}%")
print("\n" + "="*70)

if accuracy > phase2_accuracy:
    improvement = (accuracy - phase2_accuracy) / phase2_accuracy * 100
    print(f"✓ Phase 1 is BETTER than Phase 2 by {improvement:.1f}%")
    print("→ CONCLUSION: Phase 2 caused a regression!")
    if accuracy > baseline_accuracy:
        print(f"→ RECOMMENDATION: Use Phase 1 as final model (better than baseline)")
    else:
        print(f"→ WARNING: Phase 1 still worse than baseline ({baseline_accuracy*100:.2f}%)")
else:
    print(f"✗ Phase 1 is worse than or equal to Phase 2")
    print("→ CONCLUSION: Problem started before Phase 2")

print("="*70)

## 8. Detailed Classification Report

In [ ]:
# Classification report
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70 + "\n")

report = classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=3,
    zero_division=0
)
print(report)

# Save report
with open(output_dir / 'classification_report.txt', 'w') as f:
    f.write(f"Phase 1 Evaluation Results\n")
    f.write(f"Accuracy: {accuracy*100:.2f}%\n")
    f.write(f"F1-Score: {f1:.3f}\n\n")
    f.write(report)

print(f"\nReport saved to: {output_dir / 'classification_report.txt'}")

## 9. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title(f'Phase 1 Confusion Matrix\nAccuracy: {accuracy*100:.2f}%', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save
plt.savefig(output_dir / 'confusion_matrix_phase1.png', dpi=300, bbox_inches='tight')
print(f"Confusion matrix saved to: {output_dir / 'confusion_matrix_phase1.png'}")
plt.show()

## 10. Per-Class Analysis

In [ ]:
# Calculate per-class metrics
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1_per_class, support = precision_recall_fscore_support(
    all_labels,
    all_preds,
    labels=list(range(5)),
    zero_division=0
)

print("\n" + "="*70)
print("PER-CLASS METRICS")
print("="*70 + "\n")

for i, class_name in enumerate(class_names):
    print(f"{class_name}:")
    print(f"  Precision: {precision[i]:.3f}")
    print(f"  Recall:    {recall[i]:.3f}")
    print(f"  F1-Score:  {f1_per_class[i]:.3f}")
    print(f"  Support:   {support[i]}")
    print()

## 11. Sample Predictions Analysis

In [ ]:
# Show some example predictions
print("\n" + "="*70)
print("SAMPLE PREDICTIONS (First 10)")
print("="*70 + "\n")

for i in range(min(10, len(all_responses))):
    true_class = class_names[all_labels[i]]
    pred_class = class_names[all_preds[i]] if all_preds[i] != -1 else "PARSE_ERROR"
    correct = "✓" if all_labels[i] == all_preds[i] else "✗"
    
    print(f"Sample {i+1}: {correct}")
    print(f"True:      {true_class}")
    print(f"Predicted: {pred_class}")
    print(f"Response:  {all_responses[i][:200]}...")
    print("-" * 70 + "\n")

## 12. Save Complete Results

In [ ]:
# Save detailed results
results = {
    'phase': 'phase1_projector',
    'checkpoint': str(phase1_checkpoint_dir),
    'test_size': len(all_preds),
    'metrics': {
        'accuracy': float(accuracy),
        'f1_weighted': float(f1),
        'parsing_errors': int(parsing_errors),
        'parsing_error_rate': float(parsing_errors / len(all_preds))
    },
    'per_class_metrics': {
        class_names[i]: {
            'precision': float(precision[i]),
            'recall': float(recall[i]),
            'f1_score': float(f1_per_class[i]),
            'support': int(support[i])
        }
        for i in range(5)
    },
    'comparison': {
        'baseline': 0.40,
        'phase1': float(accuracy),
        'phase2': 0.1252
    },
    'confusion_matrix': cm.tolist()
}

# Save JSON
with open(output_dir / 'phase1_evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_dir / 'phase1_evaluation_results.json'}")

# Save predictions
predictions_data = [
    {
        'true_label': int(all_labels[i]),
        'predicted_label': int(all_preds[i]),
        'true_class': class_names[all_labels[i]],
        'predicted_class': class_names[all_preds[i]] if all_preds[i] != -1 else "PARSE_ERROR",
        'response': all_responses[i],
        'correct': all_labels[i] == all_preds[i]
    }
    for i in range(len(all_labels))
]

with open(output_dir / 'predictions.json', 'w') as f:
    json.dump(predictions_data, f, indent=2)

print(f"Predictions saved to: {output_dir / 'predictions.json'}")

## 13. Final Diagnostic Summary

In [ ]:
print("\n" + "="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70 + "\n")

print(f"Baseline (zero-shot):     {baseline_accuracy*100:6.2f}%")
print(f"Phase 1 (projector only): {accuracy*100:6.2f}%")
print(f"Phase 2 (LoRA):           {phase2_accuracy*100:6.2f}%")
print()

if accuracy > phase2_accuracy:
    print("✓ DIAGNOSIS: Phase 2 caused a regression")
    print()
    if accuracy > baseline_accuracy:
        improvement = (accuracy - baseline_accuracy) / baseline_accuracy * 100
        print(f"✓ Phase 1 improved over baseline by {improvement:.1f}%")
        print()
        print("RECOMMENDATION:")
        print("1. Use Phase 1 as the final model")
        print("2. Investigate why Phase 2 (LoRA) caused regression")
        print("3. Possible causes:")
        print("   - Learning rate too low (5e-5 vs 1e-3 in Phase 1)")
        print("   - Class imbalance not handled in Phase 2")
        print("   - Catastrophic forgetting")
        print("   - LoRA adapters not properly trained")
    else:
        print(f"⚠ Phase 1 still {(baseline_accuracy - accuracy) / baseline_accuracy * 100:.1f}% worse than baseline")
        print()
        print("RECOMMENDATION:")
        print("1. Both phases performed poorly")
        print("2. Consider using baseline model instead")
        print("3. Review entire training strategy")
else:
    print("✗ DIAGNOSIS: Problem started in Phase 1 or earlier")
    print()
    print("RECOMMENDATION:")
    print("1. Review Phase 1 training configuration")
    print("2. Check dataset quality and prompts")
    print("3. Consider reverting to baseline model")

print("\n" + "="*70)
print(f"\nAll results saved to: {output_dir}")
print("="*70)

## 14. Cleanup

In [ ]:
# Free memory
del model_phase1
torch.cuda.empty_cache()
gc.collect()

print("Memory cleaned up")
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")